<a href="https://colab.research.google.com/github/vignesh-potharaj/gen-ai/blob/main/Telehealth_Consultations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install required dependencies
!pip install -q google-genai pandas tabulate

In [ ]:
# Cell 2: Import modules and set up the API client
import os
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata

# Retrieve Google Gemini API key securely from Colab Secrets
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=API_KEY)
    print("✅ Gemini API Client initialized successfully.")
except Exception as e:
    print(f"❌ Error initializing client. Ensure 'GEMINI_API_KEY' is set in Colab Secrets: {e}")

✅ Gemini API Client initialized successfully.


In [ ]:
# Cell 3: Execute Naive User Query
USER_QUERY = "I've had a persistent dry cough and some light dizziness for 3 days. Is it safe for me to take standard OTC decongestants like Sudafed?"

print("--- RUNNING BASELINE (NAIVE PROMPT) ---")

# Naive call: Direct pass without patient context or safety rules
baseline_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=USER_QUERY
)

print("\n--- BASELINE OUTPUT ---")
print(baseline_response.text)

--- RUNNING BASELINE (NAIVE PROMPT) ---

--- BASELINE OUTPUT ---
Given that you're experiencing **light dizziness**, it's generally **not recommended to take standard OTC decongestants like Sudafed (pseudoephedrine) without first consulting a doctor.**

Here's why:

1.  **Dizziness is a potential side effect of decongestants:** Medications like Sudafed can cause or worsen dizziness, lightheadedness, and even increase heart rate or blood pressure in some individuals. If you're already experiencing dizziness, taking a medication that lists it as a side effect could exacerbate your symptoms or make it harder to determine the underlying cause of your dizziness.
2.  **Sudafed doesn't treat a dry cough:** Sudafed is designed to relieve nasal and sinus congestion by constricting blood vessels. It does not directly address a dry cough. For a dry cough, a cough suppressant (like dextromethorphan) or soothing remedies are typically more appropriate. Taking Sudafed for a dry cough might not offer

In [ ]:
# Cell 4: Define System Guidelines, Patient EHR Memory, and Context Window Snippets

# 1. SYSTEM RULES (Medical & Safety Guidelines)
SYSTEM_INSTRUCTIONS = """
You are a Telehealth Clinical Assistant AI. Follow these safety guardrails strictly:
1. Always cross-reference user symptoms with their Electronic Health Record (EHR) memory.
2. Check for ACE Inhibitor-induced dry cough before assuming infectious illness.
3. Decongestants containing Pseudoephedrine (e.g., Sudafed) are STRICTLY CONTRAINDICATED in patients with Hypertension.
4. Never provide a final diagnosis; offer differential considerations and flag red flags for clinical escalation.
"""

# 2. PATIENT HISTORY (Long-Term EHR Memory)
PATIENT_MEMORY = """
[PATIENT EHR PROFILE]
Name: Jane Doe | Age: 52 | Gender: Female
Active Conditions: Primary Hypertension (Diagnosed 2021)
Current Medications: Lisinopril 20mg daily (Started 3 weeks ago)
Allergies: Penicillin (Rash)
Vitals (Last recorded yesterday): BP 138/88 mmHg, HR 72 bpm
"""

# 3. CONTEXT WINDOW (Short-Term Dialogue & Recent Snippets)
CONTEXT_WINDOW = """
[RECENT CONSULTATION TRANSCRIPT]
User (2 days ago): "Hi, I started taking my new blood pressure pill Lisinopril as directed."
Assistant: "Great! Please monitor for any side effects such as dizziness or a tickly dry cough."
User (Yesterday): "Blood pressure reading was 138/88 last night."
"""

In [ ]:
# Cell 5: Run Context-Engineered Pipeline

# Constructing the structured payload combining Memory + Context + Query
engineered_prompt = f"""
{PATIENT_MEMORY}

{CONTEXT_WINDOW}

[CURRENT USER QUERY]
User: "{USER_QUERY}"
"""

print("--- RUNNING CONTEXT-ENGINEERED PROMPT ---")

context_engineered_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=engineered_prompt,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTIONS,
        temperature=0.2 # Low temperature for consistent clinical adherence
    )
)

print("\n--- CONTEXT-ENGINEERED OUTPUT ---")
print(context_engineered_response.text)

--- RUNNING CONTEXT-ENGINEERED PROMPT ---

--- CONTEXT-ENGINEERED OUTPUT ---
Thank you for reaching out. I've reviewed your symptoms and your Electronic Health Record.

Regarding your question about Sudafed: **No, it is not safe for you to take standard OTC decongestants like Sudafed.** Sudafed (Pseudoephedrine) is strictly contraindicated for patients with hypertension, as it can dangerously raise blood pressure.

Regarding your symptoms:
*   **Persistent dry cough:** Given that you started Lisinopril 3 weeks ago, a persistent dry, tickly cough is a very common side effect of ACE inhibitors like Lisinopril. This was even mentioned as a potential side effect when you started the medication.
*   **Light dizziness:** Dizziness can also be a side effect of Lisinopril, especially when starting the medication or as your blood pressure adjusts.

These symptoms are highly suggestive of side effects from your new medication, Lisinopril. It is important to discuss these with your healthcare pro